In [1]:
import os, sys

# Make sure we're in the project root
os.chdir("/Users/danesh/Documents/GitHub/trueQ")

# Ensure project root is on sys.path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from config import *
from noise_models import *
from cer import *

In [2]:
circ = random_cb_circuit(Gate.t, paulis, 10)
arr_list = effective_circuit_noise_list(circ, noisy_gates= noisy_gates_dict(sim_gate_dep[4][0]))
def offdiag_magnitude_sum(A: np.ndarray) -> float:
    # Zero the diagonal, then sum absolute values
    off = A.copy()
    np.fill_diagonal(off, 0)
    return (np.linalg.norm(off)**2).sum()

# For a list of arrays:
offdiag_sums = [offdiag_magnitude_sum(A) for A in arr_list]
offdiag_sums

[0.00039146418294212993,
 1.6086511867786247e-31,
 8.471115375272896e-32,
 1.6086511867786247e-31,
 3.540861620854889e-32,
 8.471115375272896e-32,
 8.471115375272896e-32,
 8.471115375272896e-32,
 8.471115375272896e-32,
 8.471115375272896e-32,
 0.00217781824107605]

In [3]:
sim_ZXZXZ[1][0].operator(circuit = tq.Circuit([{0:Gate.x}])).mat()

array([[ 7.80625564e-17-0.01118104j,  5.55111512e-17-0.99993749j],
       [-5.55111512e-17-0.99993749j,  7.80625564e-17+0.01118104j]])

In [12]:
noisy_sim = sim_gate_ind[4][0]

In [15]:
circ = random_cb_circuit(Gate.h, paulis, 4)
eff_circ = effective_circuit(circ, noisy_gates= noisy_gates_dict(noisy_sim))
rc_circ = rc_circuit(circuit=circ, noisy_sim= noisy_sim, n_randomizations=2000)
superop_circ = tqm.Superop.from_unitary(noisy_sim.operator(circuit=circ).mat())

rc_fid = (rc_circ@superop_circ.adj).fidelity
eff_fid = (eff_circ@superop_circ.adj).fidelity

In [16]:
rc_fid

0.9849542745709996

In [18]:
eff_fid

0.9844493614107046

In [6]:
noise_test = noisy_gates_dict(sim_ZXZXZ[4][0])

In [40]:
dic = {}

for g in [Gate.h, Gate.t, Gate.cnot]:
    dic[g] = tq.Gate(sim_ZXZXZ[4][0].operator(circuit = tq.Circuit([{0:g}])).mat())

for easy in easy_gates:
    dic[easy] = tq.Gate(sim_ZXZXZ[4][0].operator(circuit = tq.Circuit([{0:easy}])).mat())

for (p1, p2), gate in two_q_easy_gates.items():
        dic[two_q_easy_gates[(p1, p2)]] = tq.Gate(sim_ZXZXZ[4][0].operator(circuit = tq.Circuit([{0:gate}])).mat())

In [42]:
for key in dic.keys():
    print(key)
    print (dic[key] == noise_test[key])

Gate.h
True
Gate.t
True
Gate.cx
True
Gate.x
True
Gate.y
True
Gate.z
True
Gate.id
True
Gate.s
True
Gate.cliff8
True
Gate.cliff10
True
Gate.cliff11
True
Gate(XX)
False
Gate(XY)
False
Gate(XZ)
False
Gate(XI)
False
Gate(XZ, IZ)
False
Gate(XZ, IZ)
False
Gate(XY, XX)
False
Gate(XY, XX)
False
Gate(YX)
False
Gate(YY)
False
Gate(YZ)
False
Gate(YI)
False
Gate(YZ, IZ)
False
Gate(YZ, IZ)
False
Gate(YY, YX)
False
Gate(YY, YX)
False
Gate(ZX)
False
Gate(ZY)
False
Gate(ZZ)
True
Gate(ZI)
True
Gate(ZZ, IZ)
True
Gate(ZZ, IZ)
True
Gate(ZY, ZX)
False
Gate(ZY, ZX)
False
Gate(IX)
False
Gate(IY)
False
Gate(IZ)
True
Gate(II)
True
Gate(IZ)
True
Gate(IZ)
True
Gate(IY, IX)
False
Gate(IY, IX)
False
Gate(ZX, ZI)
False
Gate(ZY, ZI)
False
Gate(ZZ, ZI)
True
Gate(ZI)
True
Gate(ZI, IZ)
True
Gate(ZI, IZ)
True
Gate(ZY, ZX, ...)
False
Gate(IY, IX, ...)
False
Gate(ZX, ZI)
False
Gate(ZY, ZI)
False
Gate(ZZ, ZI)
True
Gate(ZI)
True
Gate(ZI, IZ)
True
Gate(ZI, IZ)
True
Gate(IY, IX, ...)
False
Gate(IY, IX, ...)
False
Gate(YX, XX)


In [ ]:
circ = random_cb_circuit(Gate.h, paulis, 4)
eff_circ = effective_circuit(circ, noisy_gates= noisy_gates_dict(noisy_sim))
rc_circ = rc_circuit(circuit=circ, noisy_sim= noisy_sim, n_randomizations=2000)
superop_circ = tqm.Superop.from_unitary(noisy_sim.operator(circuit=circ).mat())

rc_fid = (rc_circ@superop_circ.adj).fidelity
eff_fid = (eff_circ@superop_circ.adj).fidelity

In [19]:
import csv
import numpy as np

def run_fidelity_sweep(
    noisy_sim,
    depths=(4, 8, 12, 16, 20),
    n_repeats=5,
    csv_path="fidelity_comparison.csv",
):
    fieldnames = [
        "n_dressed_cycles",
        "effective_circuit_fidelity",
        "rc_circuit_fidelity",
        "relative_error",
    ]

    with open(csv_path, mode="w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for n in depths:
            for rep in range(n_repeats):
                circ = random_cb_circuit(Gate.h, paulis, n)
                eff_circ = effective_circuit(circ, noisy_gates=noisy_gates_dict(noisy_sim))
                rc_circ = rc_circuit(circuit=circ, noisy_sim=noisy_sim, n_randomizations=2000)
                superop_circ = tqm.Superop.from_unitary(noisy_sim.operator(circuit=circ).mat())

                rc_fid = (rc_circ @ superop_circ.adj).fidelity
                eff_fid = (eff_circ @ superop_circ.adj).fidelity
                rel_err = abs(rc_fid - eff_fid) / rc_fid if rc_fid != 0 else np.nan

                row = {
                    "n_dressed_cycles": n,
                    "effective_circuit_fidelity": float(eff_fid),
                    "rc_circuit_fidelity": float(rc_fid),
                    "relative_error": float(rel_err),
                }
                writer.writerow(row)

                print(
                    f"n={n}, repeat {rep+1}/{n_repeats}: "
                    f"rc_fid={rc_fid:.6g}, eff_fid={eff_fid:.6g}, rel_err={rel_err:.3g}"
                )

# Example call (uses your existing noisy_sim)
run_fidelity_sweep(noisy_sim=sim_gate_ind[4][0])

n=4, repeat 1/5: rc_fid=0.973215, eff_fid=0.973155, rel_err=6.13e-05
n=4, repeat 2/5: rc_fid=0.973564, eff_fid=0.97317, rel_err=0.000405
n=4, repeat 3/5: rc_fid=0.952895, eff_fid=0.9526, rel_err=0.000309
n=4, repeat 4/5: rc_fid=0.975222, eff_fid=0.975094, rel_err=0.000131
n=4, repeat 5/5: rc_fid=0.964085, eff_fid=0.9638, rel_err=0.000295
n=8, repeat 1/5: rc_fid=0.950866, eff_fid=0.950882, rel_err=1.71e-05
n=8, repeat 2/5: rc_fid=0.947784, eff_fid=0.949244, rel_err=0.00154
n=8, repeat 3/5: rc_fid=0.904372, eff_fid=0.907662, rel_err=0.00364
n=8, repeat 4/5: rc_fid=0.940172, eff_fid=0.940075, rel_err=0.000103
n=8, repeat 5/5: rc_fid=0.959229, eff_fid=0.960251, rel_err=0.00107
n=12, repeat 1/5: rc_fid=0.872312, eff_fid=0.872539, rel_err=0.00026
n=12, repeat 2/5: rc_fid=0.94499, eff_fid=0.944813, rel_err=0.000188
n=12, repeat 3/5: rc_fid=0.956387, eff_fid=0.95598, rel_err=0.000425
n=12, repeat 4/5: rc_fid=0.871707, eff_fid=0.871643, rel_err=7.32e-05
n=12, repeat 5/5: rc_fid=0.904317, eff_fi